# Упражнение 2, част 3: изображения и линейна класификация

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Част 1. Изображения като масиви](#chast-1)
5. [Част 2. Работа с изображения](#chast-2)
6. [Част 3. От изображение към признаци](#chast-3)
7. [Част 4. Линеен класификатор](#chast-4)
8. [Част 5. Промени във входното изображение](#chast-5)
9. [Част 6. Конволюция: когато съседите имат значение](#chast-6)
10. [Обобщение](#obobshtenie)
11. [Задача](#zadacha)

<a id="cel"></a>
## Цел на упражнението

В част 1 всеки **пример** беше едно цвете, описано с четири числа (дължина
и ширина на чашелистчето и венчелистчето). Моделът получаваше тези четири
числа и предсказваше вида на цветето.

Тук примерът е **изображение** на ръкописна цифра. Всяка картинка е файл
с 28 реда и 28 колони от пиксели. Компютърът не "вижда" цифрата, а получава
784 числа (28 x 28) - яркостта на всеки пиксел. Задачата е от тези числа
да се определи коя цифра е изписана.

| | Част 1 (ирис) | Част 3 (цифри) |
|---|---|---|
| Какво е един пример? | Едно цвете | Едно изображение |
| Какви са признаците? | 4 измервания (cm) | 784 пиксела (яркост 0-1) |
| Какъв е етикетът? | Вид цвете (setosa, ...) | Цифра (0, 1, ..., 9) |
| Какъв е моделът? | Perceptron | Perceptron |

Ще открием файловете от папки, ще заредим имената и етикетите в `DataFrame`,
ще прочетем изображенията като масиви и ще подготвим признаци. След това
ще използваме познатите от част 1 `fit` и `predict`.

Цифрите `0` до `9` са **имена на класове**. Предсказваме един от десет класа,
а не число като `3.7`. Това е класификация, не регресия.

Перцептронът е най-простият линеен класификатор (Розенблат, 1958) и прототипът на изкуствения неврон.

1. Смята претеглена сума на признаците плюс константа:

z = w0*x0 + w1*x1 + ... + w783*x783 + b

Ако z > 0, отговорът е "този клас", иначе "не е този клас".

2. Обучение. Минава през тренировъчните примери един по един:
- ако отговорът е верен, нищо не се променя;
- ако е грешен, теглата се коригират към верния отговор: w = w + y*x и b = b + y, където y е +1 или -1.

Така следващия път оценката за този пример е по-близо до правилната страна. Едно минаване през всички примери е епоха.

3. Ограничения.
- Границата между класовете е хиперравнина, затова моделът е линеен.
- Оценките не са вероятности.
- Ако класовете не могат да се разделят с хиперравнина, грешките никога не стават нула.

<a id="failove"></a>
## Файлове в упражнението

Работим с подмножество от **MNIST** (Modified National Institute of Standards
and Technology). Наборът съдържа 1500 изображения, по 150 за всяка цифра.
Всяко изображение е PNG файл с размер 28x28 пиксела, сиво.

Файловете са подредени в папки по цифра:

```
data/digits_png/
  0/  digit_0_0000.png ... digit_0_0149.png
  1/  digit_1_0000.png ... digit_1_0149.png
  ...
  9/  digit_9_0000.png ... digit_9_0149.png
```

Името на папката е етикетът. Имената на файловете не участват в обучението.
Всеки файл може да се отвори с обикновена програма за преглед на снимки.

| Файл | За какво служи |
|---|---|
| `Упражнение_2_част_3.ipynb` | част 3: изображения и линейна класификация |
| `data/digits_png/` | 1500 PNG изображения в 10 папки по цифра |
| `data/shapes_png/` | 600 PNG изображения в 4 папки по фигура (за задачата) |
| `ЗА_ВКЪЩИ_част_3.ipynb` | самостоятелни задачи след упражнението |

Източник: Y. LeCun, C. Cortes и C. Burges, [The MNIST Database](http://yann.lecun.com/exdb/mnist/).
Подмножеството е извлечено от [OpenML](https://www.openml.org/d/554).

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме общата среда от предишните упражнения и ядрото "Python (МС 26/27)".
Тетрадката се изпълнява самостоятелно от началото. Нужни са NumPy, Pandas,
Matplotlib и scikit-learn, които вече са включени в средата.

Стартирайте JupyterLab от папката на упражнение 2, за да работят
относителните пътища към `data/digits_png/`. Отворете тази папка
с файловия мениджър, за да видите подредбата по цифри.

In [ ]:
import solution_3

%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data/digits_png") # удобно е да се зададе веднъж, защото на много места ще се ползва
plt.rcParams.update({"figure.dpi": 110, "font.size": 11}) # увеличаваме резолюцията и фона
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

<a id="chast-1"></a>
## Част 1. Изображения като масиви

### Какво е цифрово изображение?

Цифровото изображение е **правоъгълна таблица от числа**. Всяко число
отговаря на една малка точка от картинката и се нарича **пиксел** (pixel).
При сиво изображение всеки пиксел е едно число: яркостта на тази точка.

- **0.0** означава бял фон (няма мастило);
- **1.0** означава напълно черно (максимално мастило);
- стойности между тях са нюанси на сивото (напр. 0.4 е светлосиво).

Изображение с 28 реда и 28 колони съдържа 28 x 28 = 784 пиксела. Когато
файлът се зареди в Python, той става двумерен масив (матрица) с размерност
`(28, 28)` и стойности от 0.0 до 1.0.

Машината не "вижда" картинка. Тя получава 784 числа и работи само с тях.
Не знае кое е горе и кое е долу, нито че числата образуват изображение.
Задачата на модела е да открие закономерност между тези числа и етикета.

### 1. Откриване на файловете и анализ на съдържанието им

Файловете са подредени в папки с имена `0`, `1`, ..., `9`. Името на папката
е етикетът на цифрата. `Path.rglob("*.png")` обхожда всички подпапки
и намира PNG файловете.

От пътищата извличаме:
- `filepath` - пълният път до файла;
- `filename` - само името на файла;
- `label` - цифрата, извлечена от името на папката.

Резултатът е `DataFrame`, в който всеки ред описва едно изображение.

In [ ]:
paths = sorted(DATA_DIR.rglob("*.png"))
print(f"Намерени файлове: {len(paths)}")
print(f"Пример: {paths[0]}")

df = pd.DataFrame({
    "filepath": [str(p) for p in paths],
    "filename": [p.name for p in paths],
    "label": [int(p.parent.name) for p in paths],
})
print(f"\nТаблица: {df.shape[0]} реда, {df.shape[1]} колони")
display(df.head())
print("\nБрой по цифра:")
print(df["label"].value_counts().sort_index().to_dict())

### 2. Зареждане на изображенията

`plt.imread(path)` прочита PNG файл и връща двумерен масив. За сиво
изображение 28x28 формата е `(28, 28)` със стойности от 0.0 до 1.0.
По-високата стойност означава по-тъмен щрих.

Зареждаме всички изображения в един масив `images` с форма `(1500, 28, 28)`.
Етикетите `y` идват от колоната `label` на таблицата.

In [ ]:
images = np.array([plt.imread(p) for p in df["filepath"]])
y = df["label"].values
print("Изображения:", images.shape, "dtype:", images.dtype)
print("Етикети:", y.shape)
print("Стойности на пикселите:", images.min(), "до", images.max())
print("Класове:", np.unique(y))

image_index = np.random.randint(len(images))
sample_image = images[image_index]
print(f"\nИзображение {image_index}, етикет {y[image_index]}, файл: {df['filename'].iloc[image_index]}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sample_image, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
ax.set(xlabel="Колона", ylabel="Ред",
       title=f"Изображение {image_index}, етикет {y[image_index]}")
plt.show()

Човекът вижда цифра. Имаме таблица от 28 x 28 = 784 числа.
Всяко число е яркостта на един пиксел:
0.0 означава бял фон, а 1.0 означава напълно тъмен щрих. Стойностите
между тях са нюанси на сивото.

Следващата съпка е да преобразуваме матрицата във вектор.

Моделът ще работи единствено с тези числа. Не знае, че идват от картинка,
не знае, че е цифра, и не знае кои пиксели са съседни. Единственото, което
получава, е ред от 784 стойности и етикет (верния отговор).

### 3. Галерия от примери

Следващата помощна функция само подрежда картинки и надписи. Ще я използваме
и при разглеждането на предсказанията. `vmin=0` и `vmax=1` запазват еднаква
цветова скала за всички изображения.

In [ ]:
def show_images(selected_images, titles, ncols=5):
    nrows = (len(selected_images) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.3 * ncols, 2.6 * nrows),
                             squeeze=False, layout="constrained")
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, title in zip(axes.ravel(), selected_images, titles):
        ax.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title, fontsize=10)
    plt.show()

indexes = np.random.choice(len(images), size=15, replace=False)
show_images(images[indexes], [f"Позиция {i}, етикет {y[i]}" for i in indexes])

### Галерия по класове

Следващата клетка показва по четири примера за всяка цифра от 0 до 9.

In [ ]:
n_per_class = 4
fig, axes = plt.subplots(10, n_per_class, figsize=(2.3 * n_per_class, 2.3 * 10), layout="constrained")
for digit in range(10):
    class_images = images[y == digit]
    for col in range(n_per_class):
        ax = axes[digit, col]
        ax.imshow(class_images[col], cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
        ax.axis("off")
        if col == 0:
            ax.set_title(f"Цифра {digit}", fontsize=11, fontweight="bold", loc="left")
fig.suptitle("По четири примера за всяка цифра", fontsize=13, y=1.005)
plt.show()

### Средна цифра по клас

Ако осредним всички изображения с един и същ етикет, получаваме размазана
версия на тази цифра. Пикселите, които винаги са тъмни (напр. горната
дъга на "0"), остават тъмни, а тези, които се различават между почерците,
стават сиви. Средното изображение показва кои позиции са типични за всяка
цифра.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5.5), layout="constrained")
for digit, ax in enumerate(axes.ravel()):
    mean_image = images[y == digit].mean(axis=0)
    ax.imshow(mean_image, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    count = int((y == digit).sum())
    ax.set_title(f"Цифра {digit}\n({count} примера)", fontsize=10)
    ax.axis("off")
fig.suptitle("Средно изображение за всяка цифра", fontsize=13)
plt.show()

### Разглеждане на произволно изображение

Следващата функция показва изображение на избрана позиция заедно с числовите
стойности на пикселите и хистограма на разпределението им. Хистограмата
показва колко пиксела попадат във всеки диапазон: повечето са близо до 0
(бял фон), а малка част са близо до 1 (тъмен щрих).

In [ ]:
def show_digit(idx):
    """Показва изображение, числовите му стойности и хистограма."""
    img, label = images[idx], y[idx]
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    ax1.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    ax1.set(xlabel="Колона", ylabel="Ред",
            title=f"Позиция {idx}, етикет {label}")
    # ВАЖНО: !!!
    # img е масив с форма (28, 28) - матрица.
    # reshape(-1) го превръща в едномерен масив с форма (784,).
    # Аргументът -1 означава "изчисли размера автоматично" (28 * 28 = 784).
    # Същото като img.flatten(), но reshape връща view 
    # (без копиране на паметта), а flatten винаги копира.
    flat = img.reshape(-1)
    ax2.bar(range(len(flat)), flat, color="#168c81", width=0.8)
    ax2.set(xlabel="Номер на признака (0-783)", ylabel="Стойност",
            title="784-те стойности като стълбчета")
    ax2.set_xlim(-1, len(flat))
    # Хистограмата групира стойностите в интервали (bins) и показва колко стойности попадат във всеки интервал.
    # Тук виждаме, че повечето пиксели са ~0 (фон), а малка част са ~1 (щрих).
    ax3.hist(flat, bins=30, color="#168c81", edgecolor="white")
    ax3.set(xlabel="Стойност на пиксела", ylabel="Брой пиксели",
            title="Разпределение на стойностите")
    ax3.axvline(flat.mean(), color="#c0392b", linestyle="--", label=f"средно: {flat.mean():.2f}")
    ax3.legend(fontsize=9)
    plt.show()

show_digit(np.random.randint(len(images)))

### Стълбчетата на всички примери за всяка цифра

Следващата клетка рисува стълбчетата на всички 150 примера от всяка цифра,
наложени едно върху друго. Всеки пример е почти прозрачен. Там, където много
примери имат висока стойност, цветът се натрупва и става тъмен. Там, където
щрих имат само няколко примера, остава бледо.

Най-тъмните ивици показват позициите, на които повечето примери имат щрих.
Идеята е същата като при средното изображение, но тук пикселите са подредени
в ред от 784 признака. Ивиците се повтарят на всеки 28 позиции, защото всеки
ред от изображението заема 28 поредни признака. Различните цифри имат ивици
на различни места и именно тези разлики моделът ще използва.

Клетката се изпълнява няколко секунди, защото рисува 1500 x 784 стълбчета.

In [ ]:
fig, axes = plt.subplots(10, 1, figsize=(14, 14), layout="constrained", sharex=True)
for digit, ax in enumerate(axes):
    class_images = images[y == digit]
    class_flat = class_images.reshape(len(class_images), -1)
    # fill_between със step="mid" рисува същите стълбчета като bar, но много по-бързо.
    # alpha=0.01: цветът се натрупва там, където много примери имат висока стойност.
    for flat in class_flat:
        ax.fill_between(range(len(flat)), flat, step="mid", color="#168c81",
                        alpha=0.01, linewidth=0)
    ax.set(xlim=(0, 783), ylim=(0, 1.05), yticks=[0, 1])
    ax.set_title(f"Цифра {digit} ({len(class_images)} примера)", loc="left", fontsize=10)
fig.supxlabel("Номер на признака (0-783)")
fig.supylabel("Стойност на пиксела")
fig.suptitle("Стълбчетата на всички примери от всяка цифра, наложени едно върху друго",
             fontsize=12)
plt.show()

In [ ]:
# --- ДА ОПИТАМЕ 1 ---
#
# Изберете всички изображения с етикет 7 чрез булева маска.
# Покажете първите шест с show_images и отпечатайте общия им брой.
# Еднакви ли са пикселите, въпреки че етикетът е един и същ?


solution_3.to_test_1(images, y, show_images)

<a id="chast-2"></a>
## Част 2. Работа с изображения

Преди обучението е полезно да се знае как едно изображение се зарежда,
разглежда и преобразува. Това е подготовка на данните, както нормализацията
и конструирането на признаци при таблиците. Всяко преобразуване
променя 784-те числа, които моделът получава.

### Зареждане с Pillow

Освен с `plt.imread`, изображенията често се зареждат с библиотеката
[Pillow](https://pillow.readthedocs.io/) (`from PIL import Image`). Разликата
е в скалата на стойностите:
- `plt.imread` връща дробни числа от 0.0 до 1.0;
- `np.array(Image.open(...))` връща цели числа `uint8` от 0 до 255.

За да се получи същата скала, стойностите от Pillow се делят на 255. Това е
нормализация: всички признаци попадат между 0 и 1.

`.mode` показва вида на изображението: `"L"` е сиво (един канал), а `"RGB"`
е цветно (три канала: червено, зелено и синьо) с форма
`(височина, ширина, 3)`. `.convert("L")` превръща цветно изображение в сиво,
а `.resize((28, 28))` променя размера. Така може да се подготви и собствена
снимка на цифра.

Досега 0 изглеждаше като бял фон, а 1 като черен щрих. Това е само начинът
на показване. Във файла 0 е черно, а 255 е бяло, затова програмата за снимки
показва бяла цифра върху черен фон. `cmap="gray_r"` обръща цветовете, за да
прилича на мастило върху хартия. Преди да се подаде на модела, тя трябва да се обърне с `1 - image`.

In [ ]:
from PIL import Image

example_index = 621
pil_image = Image.open(df["filepath"].iloc[example_index])
print("Режим:", pil_image.mode, "| размер:", pil_image.size)
pil_array = np.array(pil_image)
print("Pillow:", pil_array.dtype, pil_array.min(), "до", pil_array.max())
print("plt.imread:", images[example_index].dtype, images[example_index].min(), "до",
      images[example_index].max())
print("Еднакви след деление на 255:", np.allclose(pil_array / 255, images[example_index]))

fig, axes = plt.subplots(1, 2, figsize=(6, 3.4), layout="constrained")
axes[0].imshow(pil_array, cmap="gray", interpolation="nearest")
axes[0].set_title('cmap="gray" (както във файла)', fontsize=10)
axes[1].imshow(pil_array, cmap="gray_r", interpolation="nearest")
axes[1].set_title('cmap="gray_r" (в тетрадката)', fontsize=10)
for ax in axes:
    ax.set(xticks=[], yticks=[])
plt.show()

### Анализ: рамка и център на масата

Две прости характеристики описват къде е цифрата в изображението:
- **рамка** (bounding box): първият и последният ред и колона, в които има
  поне един ненулев пиксел;
- **център на масата**: средният ред и средната колона на пикселите,
  претеглени с яркостта им. Тъмните пиксели "тежат" повече, а фонът не
  тежи нищо.

Средата на изображение 28x28 е точката (13.5, 13.5), между четирите
централни пиксела. При създаването на MNIST всяка цифра е преместена така,
че центърът на масата ѝ да е в средата. Затова един и същ пиксел има сходна
роля в различните изображения. Последният ред на клетката проверява центровете на всички
1500 изображения.

In [ ]:
def center_of_mass(img):
    """Средният ред и средната колона, претеглени с яркостта на пикселите."""
    total = img.sum()
    row = (img.sum(axis=1) * np.arange(img.shape[0])).sum() / total
    col = (img.sum(axis=0) * np.arange(img.shape[1])).sum() / total
    return row, col

example = images[example_index]
ink_rows = np.flatnonzero(example.max(axis=1) > 0)
ink_cols = np.flatnonzero(example.max(axis=0) > 0)
top, bottom, left, right = ink_rows[0], ink_rows[-1], ink_cols[0], ink_cols[-1]
center_row, center_col = center_of_mass(example)
print(f"Етикет: {y[example_index]}")
print(f"Дял на тъмните пиксели (над 0.5): {(example > 0.5).mean():.2f}")
print(f"Рамка: редове от {top} до {bottom}, колони от {left} до {right}")
print(f"Център на масата: ред {center_row:.1f}, колона {center_col:.1f}")

fig, ax = plt.subplots(figsize=(4.6, 4.6), layout="constrained")
ax.imshow(example, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
ax.add_patch(plt.Rectangle((left - 0.5, top - 0.5), right - left + 1, bottom - top + 1,
                           fill=False, edgecolor="#168c81", linewidth=2))
ax.plot(center_col, center_row, "x", color="#c0392b", markersize=12, markeredgewidth=3)
ax.plot(13.5, 13.5, "+", color="#1f4e79", markersize=16, markeredgewidth=2)
ax.set(xticks=[], yticks=[], title="Рамка, център на масата (x)\nи среда на изображението (+)")
plt.show()

centers = np.array([center_of_mass(img) for img in images])
print(f"Центрове на всички {len(images)} изображения: ред от {centers[:, 0].min():.1f}"
      f" до {centers[:, 0].max():.1f}, колона от {centers[:, 1].min():.1f}"
      f" до {centers[:, 1].max():.1f}")

### Преобразувания

Често използвани преобразувания, всяко с един ред код:
- **инверсия** `1 - example`: обръща светло и тъмно;
- **праг** `example > 0.5`: оставя само черно и бяло;
- **изрязване** по рамката: премахва празните краища, но променя размера;
- **огледално обръщане** `np.fliplr` и **завъртане** `np.rot90`;
- **по-ниска резолюция**: `reshape(14, 2, 14, 2)` разделя изображението на
  блокове 2x2, а `mean(axis=(1, 3))` заменя всеки блок със средното му.
  Остават 14 x 14 = 196 признака вместо 784.

Над всяко изображение е показана формата му. Модел, обучен с 784 признака,
приема само изображения 28x28, затова изрязаното или намаленото изображение
трябва отново да се приведе до този размер.

In [ ]:
cropped = example[top:bottom + 1, left:right + 1]
low_res = example.reshape(14, 2, 14, 2).mean(axis=(1, 3))
transformed = [example, 1 - example, (example > 0.5).astype(float), cropped,
               np.fliplr(example), np.rot90(example, 2), low_res]
titles = ["Оригинал", "Инверсия", "Праг 0.5", "Изрязване",
          "Огледално", "Завъртане 180°", "По-ниска резолюция"]
show_images(transformed, [f"{t}\n{img.shape}" for t, img in zip(titles, transformed)],
            ncols=4)

### Центриране

Ако цифрата е изместена, тя може да се върне в средата: изчислява се
центърът на масата и изображението се измества така, че той да попадне
в (13.5, 13.5). `np.roll` измества масив по редове и колони. Пикселите,
които излизат от едната страна, влизат от другата, но краищата на това
изображение са празни, затова нищо не се губи.

В т. 10 се вижда, че моделът може да сбърка, когато цифрата е изместена
само с три колони. Центрирането преди предсказване е един начин да се
избегне това.

In [ ]:
off_center = np.roll(example, (-3, 4), axis=(0, 1))
off_row, off_col = center_of_mass(off_center)
centered = np.roll(off_center, (round(13.5 - off_row), round(13.5 - off_col)), axis=(0, 1))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.8), layout="constrained")
for ax, img, title in zip(axes, [example, off_center, centered],
                          ["Оригинал", "Изместено", "Центрирано"]):
    r, c = center_of_mass(img)
    ax.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    ax.plot(c, r, "x", color="#c0392b", markersize=12, markeredgewidth=3)
    ax.plot(13.5, 13.5, "+", color="#1f4e79", markersize=16, markeredgewidth=2)
    ax.set(xticks=[], yticks=[], title=f"{title}\nцентър ({r:.1f}, {c:.1f})")
plt.show()

<a id="chast-3"></a>
## Част 3. От изображение към признаци

Линейният класификатор очаква данните като **таблица**: всеки ред е един
пример, всяка колона е един признак. При ириса таблицата имаше 5 колони.
Тук ще има 784 колони - по една за всеки пиксел.

Изображението обаче е двумерен масив 28x28. За да го подадем на модела,
трябва да го "разгънем" в един дълъг ред от 784 числа. Тази операция се
нарича `reshape`. Тя не губи информация: може да се обърне обратно.

### 4. 28 реда по 28 числа като един ред от 784 числа

`reshape(-1)` подрежда стойностите на едно изображение в едномерен масив.
При стандартния ред на NumPy първо идват 28-те числа от ред 0, след тях
28-те числа от ред 1 и т.н. `-1` означава NumPy да изчисли този размер.
Нито една стойност не се изтрива или усреднява.

In [ ]:
flat_image = sample_image.reshape(-1)
print("Преди:", sample_image.shape, "след:", flat_image.shape)
print("Възстановяване без промяна:",
      np.array_equal(flat_image.reshape(28, 28), sample_image))

Пикселът на позиция `(ред, колона)` става признак с номер `28 * ред + колона`.
Например пикселът `(10, 14)` - приблизително средата на изображението - е
признак номер `28 * 10 + 14 = 294`.

Следващата клетка взема 20 пиксела от ред 10 (колони от 4 до 23) и ги сравнява
със съответните признаци от `flat_image`.

In [ ]:
row = 10
cols = np.arange(4, 24)
feature_numbers = 28 * row + cols
excerpt = pd.DataFrame({
    "ред": row,
    "колона": cols,
    "номер = 28 * ред + колона": feature_numbers,
    "sample_image[ред, колона]": sample_image[row, cols],
    "flat_image[номер]": flat_image[feature_numbers],
})
display(excerpt.round(3))
print("Всички 20 стойности съвпадат:",
      np.array_equal(sample_image[row, cols], flat_image[feature_numbers]))

### 5. Матрицата с признаци `X` и вектор с етикети `y`

`reshape(len(images), -1)` запазва броя изображения и събира пикселите на
всяко изображение в един ред. Получаваме форма `(1500, 784)`.

Сега данните имат същия вид като при ириса:

| | Ирис | Цифри |
|---|---|---|
| `X` | `(150, 4)` - 150 цветя по 4 измервания | `(1500, 784)` - 1500 картинки по 784 пиксела |
| `y` | `(150,)` - вид цвете | `(1500,)` - коя цифра |

**`X` съдържа само пикселите** (входните данни за модела).
**`y` съдържа само етикетите** (верните отговори).
При `fit` моделът получава и двете. При `predict` получава само `X` и
трябва сам да определи `y`.

Стойностите вече са между 0 и 1, защото `plt.imread` ги прочита в тази
скала. Допълнително нормализиране не е необходимо.

In [ ]:
X = images.reshape(len(images), -1)
print("X:", X.shape, "y:", y.shape)
print("Стойности в X:", X.min(), "до", X.max())

In [ ]:
# --- ДА ОПИТАМЕ 2 ---
#
# Възстановете изображение с позиция 123 само от реда X[123].
# Използвайте reshape(28, 28) и покажете с show_images.
# Сравнете с images[123] чрез np.array_equal.
# На коя позиция в X[123] се намира пикселът от ред 10, колона 14?

solution_3.to_test_2(X, images, y, show_images)

<a id="chast-4"></a>
## Част 4. Линеен класификатор

### 6. Разделяне на данните за обучение и проверка

Моделът трябва да разпознава цифри, които **не е виждал по време на
обучението**. Ако проверяваме точността само върху картинките, от които е
учил, не знаем дали наистина е научил нещо или просто е запомнил отговорите.

Затова разделяме данните на две части:
- **тренировъчна** (80%) - моделът учи от тези примери;
- **тестова** (20%) - проверяваме върху тях, без моделът да ги е виждал.

`stratify=y` запазва дела на десетте цифри в двете части.
`random_state=42` прави разделянето възпроизводимо.
Разделяме и масива с изображения, за да можем да ги показваме по-късно.

In [ ]:
X_train, X_test, y_train, y_test, train_images, test_images = train_test_split(
    X, y, images, test_size=0.2, random_state=42, stratify=y
)
print("За обучение:", X_train.shape, y_train.shape)
print("За проверка:", X_test.shape, y_test.shape)
print("Цифри в тренировъчната част:", np.bincount(y_train))
print("Цифри в тестовата част:", np.bincount(y_test))

### 7. Обучение с `fit`

Ще използваме [`Perceptron`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html)
като готов линеен класификатор. Имаме много класове (10-те цифри). scikit-learn обучава 10 перцептрона,
по един за всяка цифра срещу всички останали. Всеки дава своя линейна оценка, а предсказанието е цифрата
с най-голяма оценка. Затова coef_ има форма (10, 784), а decision_function връща 10 числа.
Интерфейсът е вече добре познат:

- `model.fit(X_train, y_train)` - моделът разглежда тренировъчните примери
  заедно с верните им етикети и научава коефициенти (числа), които свързват
  стойностите на пикселите с класа;
- `model.predict(X_test)` - моделът получава **само пикселите** на нови
  изображения и връща предсказания клас за всяко от тях.

За всяка възможна цифра моделът умножава всеки пиксел по научен коефициент,
събира резултатите и добавя константа. Получава по една **линейна оценка**
за всеки клас и избира класа с най-голяма оценка.

Тук използваме готовата реализация. Как точно се определят коефициентите
ще бъде разгледано през идните седмици.

In [ ]:
model = Perceptron(random_state=42)
model.fit(X_train, y_train)
print("Класове на модела:", model.classes_)

### 8. Предсказване за едно изображение

Моделът очаква двумерен вход дори за една картинка: `(1, 784)`.
`X_test[0:1]` запазва реда като двумерен масив; `X_test[0]` би имал форма `(784,)`.
При `predict` подаваме само пикселите. Етикетът е нужен след това за сравнение.

In [ ]:
one_image = X_test[0:1]
one_prediction = model.predict(one_image)[0]
print("Форма на входа:", one_image.shape)
print("Предсказание:", one_prediction)
print("Верен етикет:", y_test[0])
show_images(test_images[:1],
            [f"Вярно: {y_test[0]} | модел: {one_prediction}"], ncols=1)

### Разглеждане на произволно предсказание

Следващата функция показва тестово изображение заедно с предсказанието и
десетте линейни оценки. Зеленото стълбче е избраният клас. Пробвайте
различни стойности на `test_index`.

In [ ]:
def predict_and_explain(test_index):
    """Показва тестово изображение, предсказание и линейни оценки."""
    img = test_images[test_index]
    actual = y_test[test_index]
    row = X_test[test_index:test_index+1]
    pred = model.predict(row)[0]
    scores = model.decision_function(row)[0]
    print(scores)
    winner = model.classes_[scores.argmax()]
    correct = pred == actual
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained",
                                   gridspec_kw={"width_ratios": [1, 2]})
    ax1.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    result_text = "ВЯРНО" if correct else "ГРЕШКА"
    color = "#168c81" if correct else "#c0392b"
    ax1.set_title(f"Позиция {test_index}\nВерен: {actual} | модел: {pred}\n{result_text}",
                  fontsize=10, color=color)
    ax1.axis("off")
    bar_colors = ["#168c81" if c == winner else "#bfc8d1" for c in model.classes_]
    if not correct:
        bar_colors[actual] = "#e74c3c"
    ax2.bar(model.classes_, scores, color=bar_colors)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set(xticks=model.classes_, xlabel="Клас", ylabel="Линейна оценка",
            title="Десетте оценки на модела")
    plt.show()

predict_and_explain(np.random.randint(len(X_test)))

### 9. Предсказвания за много изображения

Един ред във входа дава едно предсказание. Показваме първите 15 тестови
изображения и сравняваме с известните етикети.

In [ ]:
predictions = model.predict(X_test)
print("Брой предсказания:", len(predictions))
print("Правилно разпознати:", int((predictions == y_test).sum()), "от", len(y_test))
show_images(test_images[:15],
            [f"Вярно: {actual} | модел: {predicted}"
             for actual, predicted in zip(y_test[:15], predictions[:15])])

Сравнението с `==` дава булев масив, а `sum()` брои стойностите `True`.
Това е най-простата мярка за качество: **точност** (accuracy) = брой правилни
/ общ брой. Тя казва колко често моделът е прав, но не казва *кои* цифри
бърка. За това служи матрицата на объркванията.

`np.flatnonzero(...)` връща позициите в тестовия масив, за които условието
е изпълнено. Грешните предсказания са интересни: показват къде моделът
се затруднява.

In [ ]:
unmatched_positions = np.flatnonzero(predictions != y_test)
print("Първи позиции с несъвпадение:", unmatched_positions[:10])
print("Общо неправилни:", len(unmatched_positions), "от", len(y_test))

In [ ]:
# --- ДА ОПИТАМЕ 3 ---
#
# Покажете първите шест неправилно разпознати изображения.
# В надписите включете верния етикет и предсказанието.
# Посочете един пример, за който разпознаването е трудно и за човек.

solution_3.to_test_3(unmatched_positions, test_images, y_test, predictions, show_images)
_fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
ax1.imshow(test_images[13], cmap="gray_r")
ax2.imshow(test_images[4], cmap="gray_r")
ax3.imshow(test_images[9], cmap="gray_r")

### Базов модел

Колко добър е този резултат? Както при ириса, полезно е сравнение с базов
модел, който изобщо не гледа пикселите. `DummyClassifier(strategy="stratified")`
избира клас случайно според честотата на класовете в тренировъчните данни.
Тук всички цифри са по равно, затова всяка има шанс 1 към 10, така че случайните
отговори са верни приблизително в 10% от случаите. `random_state=42` прави
случайните отговори едни и същи при всяко изпълнение.

Ако Perceptron е много по-добър от базовия модел, значи действително сме
успяли да обучим по-добър класификатор.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="stratified", random_state=42)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
print("Първите 10 верни етикета:", y_test[:10])
print("Първите 10 случайни отговора:", baseline_predictions[:10])
print("Perceptron:", int((predictions == y_test).sum()), "от", len(y_test), "правилни")
print("Случаен базов модел:", int((baseline_predictions == y_test).sum()), "от", len(y_test), "правилни")

### Матрица на объркванията

Матрицата на объркванията показва кои цифри се бъркат помежду си.
Всеки ред е верен етикет, всяка колона е предсказание на модела.
Числата по диагонала са правилните отговори. Число извън диагонала
означава, че тези тестови изображения са объркани с друга цифра.
Тъмните квадратчета показват по-голям брой.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions, labels=range(10))
fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color=color, fontsize=10)
ax.set(xticks=range(10), yticks=range(10),
       xlabel="Предсказан клас", ylabel="Верен клас",
       title="Матрица на объркванията")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()
print("По диагонала (правилни):", cm.diagonal().sum())
print("Извън диагонала (грешни):", cm.sum() - cm.diagonal().sum())

In [ ]:
# --- ДА ОПИТАМЕ 4 ---
#
# Намерете двойката цифри, които моделът бърка най-често.
# Използвайте матрицата cm. Занулете диагонала, за да не
# пречи, и намерете позицията на максималната стойност.
# Покажете три тестови изображения от тази двойка.

solution_3.to_test_4(cm, test_images, y_test, predictions, show_images)

<a id="chast-5"></a>
## Част 5. Промени във входното изображение

Моделът не "разбира" какво е цифра. Не знае, че "6" е обърнато "9" или
че "1" е тънка вертикална черта. Единственото, което е научил, е кои
**позиции на пикселите** обикновено са тъмни за всеки клас. Ако преместим
цифрата на друго място в картинката, стойностите попадат на други позиции
и моделът може да сбърка, въпреки че за човека цифрата е същата.

Следващите примери показват тази чувствителност.

### 10. Изместване с няколко пиксела

Какво става, ако преместим същата рисунка с три колони надясно?
Създаваме нов масив с нули и копираме първите 25 колони в последните 25.
Обученият модел остава същият.

Мрежата върху изображенията разделя отделните пиксели. Червената пунктирана
линия отбелязва левия край на цифрата в оригинала. В изместеното изображение
между линията и цифрата има точно три празни колони.

In [ ]:
original_image = test_images[0]
shift = 3
shifted_image = np.zeros_like(original_image)
shifted_image[:, shift:] = original_image[:, :-shift]

pair = np.array([original_image, shifted_image])
pair_flat = pair.reshape(2, -1)
pair_predictions = model.predict(pair_flat)

# Първата колона с поне един тъмен пиксел в оригинала.
first_column = np.flatnonzero(original_image.max(axis=0) > 0)[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5.3), layout="constrained")
for ax, img, title in zip(axes, pair, [f"Оригинал: модел {pair_predictions[0]}",
                                       f"Изместено с {shift}: модел {pair_predictions[1]}"]):
    ax.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    # Мрежата е по границите на пикселите: -0.5, 0.5, ..., 27.5.
    ax.set_xticks(np.arange(-0.5, 28), minor=True)
    ax.set_yticks(np.arange(-0.5, 28), minor=True)
    ax.grid(which="minor", color="#bfc8d1", linewidth=0.6)
    ax.tick_params(which="minor", length=0)
    ax.set_xticks(range(0, 28, 4))
    ax.set_yticks(range(0, 28, 4))
    ax.axvline(first_column - 0.5, color="#c0392b", linestyle="--", linewidth=1.5)
    ax.set(xlabel="Колона", ylabel="Ред", title=title)
plt.show()
print("Верен етикет на оригинала:", y_test[0])

При това конкретно изображение промяната на позицията може да промени
предсказанието. Разпознаването на един набор от подготвени картинки не
означава, че моделът разпознава всякакви снимки на цифри.

### 11. Собствено изображение

Следващата тройка е нарисувана ръчно в масив 28x28. Използваме същите
размери и скала като в обучението. Верният замисъл е 3, но предсказанието
не е гарантирано.

In [ ]:
handmade_image = np.zeros((28, 28), dtype=float)
handmade_image[4:8, 8:18] = 1.0
handmade_image[8:12, 16:20] = 1.0
handmade_image[12:16, 10:18] = 1.0
handmade_image[16:20, 16:20] = 1.0
handmade_image[20:24, 8:18] = 1.0

handmade_prediction = model.predict(handmade_image.reshape(1, -1))[0]
show_images([handmade_image],
            [f"Замислена цифра: 3 | модел: {handmade_prediction}"], ncols=1)

In [ ]:
# --- ДА ОПИТАМЕ 5 ---
#
# Подайте напълно празно изображение 28x28 с всички стойности 0.
# Покажете изображението и предсказания клас.
# Има ли моделът клас "няма цифра"? Какво означава полученият отговор?

solution_3.to_test_5(model)

### 12. Разбъркани редове на изображението

Какво ще стане, ако разбъркаме 28-те реда на всички изображения по един и
същ начин и обучим модела наново? `np.random.default_rng(42).permutation(28)`
дава фиксирана нова подредба на редовете, една и съща при всяко изпълнение.
`train_images[:, row_order, :]` подрежда редовете на всяко изображение в тази
нова подредба. Примерите за обучение и проверка остават същите.

За човек цифрите вече не се разпознават. Ще прави ли новият модел отлични
предсказания? Защо?

In [ ]:
row_order = np.random.default_rng(42).permutation(28)
print("Нова подредба на редовете:", row_order)
shuffled_train_images = train_images[:, row_order, :]
shuffled_test_images = test_images[:, row_order, :]
show_images(np.concatenate([test_images[:5], shuffled_test_images[:5]]),
            [f"Оригинал, етикет {label}" for label in y_test[:5]]
            + [f"Разбъркано, етикет {label}" for label in y_test[:5]])

X_shuffled_train = shuffled_train_images.reshape(len(shuffled_train_images), -1)
X_shuffled_test = shuffled_test_images.reshape(len(shuffled_test_images), -1)
shuffled_model = Perceptron(random_state=42)
shuffled_model.fit(X_shuffled_train, y_train)
shuffled_predictions = shuffled_model.predict(X_shuffled_test)
print("Стар модел, оригинални картинки:", int((predictions == y_test).sum()), "от", len(y_test))
print("Нов модел, разбъркани картинки:", int((shuffled_predictions == y_test).sum()), "от", len(y_test))

**Отговор.** Да, толкова добри, колкото и преди. Новият модел разпознава
същия брой изображения и дава същите предсказания за всички тестови примери.
Разбъркването нито помага, нито пречи.

Perceptron не знае кои пиксели са съседни. За него изображението е ред от
784 отделни признака и за всеки признак научава собствен коефициент.
Разбъркването е едно и също за всички изображения, затова всеки признак идва
винаги от един и същ пиксел, и при обучението, и при проверката. Никоя
стойност не се губи, само позициите се сменят. Затова моделът научава
същите коефициенти, разместени по същия начин.

Старият модел обаче не може да разпознава разбъркани картинки. Неговите
тегла са изчислени с признаците от оригиналните позиции, а там вече стоят други
пиксели, и той се справя почти толкова зле, колкото случайният базов модел.
Обратното също е валидно - новият модел не се справя с оригиналните картинки!

In [ ]:
print("Стар модел, разбъркани картинки:",
      int((model.predict(X_shuffled_test) == y_test).sum()), "от", len(y_test))
print("Нов модел, оригинални картинки:",
      int((shuffled_model.predict(X_test) == y_test).sum()), "от", len(y_test))

<a id="chast-6"></a>
## Част 6. Конволюция: когато съседите имат значение

Perceptron гледа всеки пиксел поотделно, затова разбъркването не му пречи.
Редът на характеристиките не е от значение за него, но едно изображение
е наредена последователност от пиксели, не е случайна подредба.
Така той не "научава" формата на цифрата. Щрихите, ръбовете и ъглите са
групи от съседни пиксели, а съседството се губи, когато изображението стане
ред от 784 отделни признака.

### 13. Филтър и карта на признаците

**Конволюцията** работи с малки области от изображението. Малка таблица от
числа, наречена **филтър** (или ядро, kernel на английски), например 3x3, се плъзга по
изображението. На всяка позиция 9-те съседни пиксела под филтъра се умножават
по 9-те числа на филтъра и се събират. Един и същ филтър се използва на всяка
позиция. Резултатът е ново изображение, наречено **карта на признаците**
(feature map). То показва къде в картинката има това, което филтърът "търси".

Функцията `convolve` по-долу прави точно това с два вложени цикъла. `np.pad`
добавя рамка от нули около изображението, за да може филтърът да се центрира
и върху крайните пиксели и картата да остане 28x28.

Двата филтъра търсят ръбове:
- `horizontal_edges` изважда трите пиксела над точката от трите под нея.
  Стойността е голяма (червено), когато отгоре е фон, а отдолу е щрих, т.е.
  в горния край на хоризонтална черта. В долния край е отрицателна (синьо).
- `vertical_edges` е същият филтър, транспониран. Сравнява лявата и дясната
  колона: червено е левият край на вертикална черта, синьо е десният.

Там, където около точката няма промяна (фон или вътрешност на щриха),
резултатът е около 0 (бяло).

In [ ]:
def convolve(image, kernel):
    """Плъзга филтър 3x3 по изображението и връща карта на признаците."""
    padded = np.pad(image, 1)
    feature_map = np.zeros(image.shape)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            neighborhood = padded[r:r + 3, c:c + 3]
            feature_map[r, c] = (neighborhood * kernel).sum()
    return feature_map

horizontal_edges = np.array([[-1, -1, -1],
                             [ 0,  0,  0],
                             [ 1,  1,  1]])
vertical_edges = horizontal_edges.T
print("Филтър за хоризонтални ръбове:\n", horizontal_edges)
print("Филтър за вертикални ръбове:\n", vertical_edges)

digit_image = test_images[0]
feature_maps = [convolve(digit_image, horizontal_edges), convolve(digit_image, vertical_edges)]
fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
axes[0].imshow(digit_image, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
axes[0].set_title(f"Изображение, етикет {y_test[0]}")
for ax, feature_map, title in zip(axes[1:], feature_maps,
                                  ["Хоризонтални ръбове", "Вертикални ръбове"]):
    im = ax.imshow(feature_map, cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
    ax.set_title(title)
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes[1:], shrink=0.8)
plt.show()

### 14. Конволюция на разбъркано изображение

Сега същият филтър се прилага към разбърканото изображение от т. 12. След
това редовете на картата на признаците се връщат в оригиналната подредба с
`np.argsort(row_order)`, както коефициентите в т. 12. При Perceptron това
даде точно същите коефициенти. Ще се получи ли същата карта и тук?

In [ ]:
original_map = convolve(test_images[0], horizontal_edges)
shuffled_map = convolve(shuffled_test_images[0], horizontal_edges)
restored_map = shuffled_map[np.argsort(row_order)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
for ax, feature_map, title in zip(axes, [original_map, shuffled_map, restored_map],
                                  ["Оригинал", "Разбъркано", "Разбъркано, върнати редове"]):
    ax.imshow(feature_map, cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()
print("Същата карта след връщане на редовете:", np.allclose(restored_map, original_map))

Не. Всяка стойност в картата зависи от 3x3 съседни пиксела. След
разбъркването съседите на почти всеки пиксел са други, затова филтърът намира
ръбове там, където няма, и пропуска истинските. Връщането на редовете не
помага, защото грешните стойности вече са изчислени. При конволюцията
подредбата на пикселите има значение, а за Perceptron нямаше никакво значение.

В **конволюционните невронни мрежи** (CNN) числата във филтрите не се задават
ръчно, а се научават от данните, както Perceptron научава своите коефициенти.
Мрежата използва много филтри и няколко слоя: първите откриват ръбове, а
следващите ги комбинират в ъгли, дъги и цели части от цифри.

### 15. Модел с конволюционни признаци

Филтрите могат да се използват като признаци за същия Perceptron. За всяко
изображение се правят четири карти на признаците: за хоризонтални,
вертикални и два вида диагонални ръбове. Положителните и отрицателните
стойности се пазят в отделни карти, защото показват противоположни краища
на щриха, например горен и долен. `np.maximum(..., 0)` занулява
отрицателните стойности. В невронните мрежи това се нарича ReLU.

След това всяка карта се намалява с **max pooling**: тя се разделя на
блокове 4x4 и от всеки блок остава само най-голямата стойност. От карта
28x28 остават 7x7 = 49 числа. Това е същият трик с `reshape` като при
по-ниската резолюция в Част 2, но с `max` вместо `mean`. Ако цифрата се
измести с пиксел-два, ръбът често остава в същия блок и признакът не се
променя.

Следващата клетка показва една карта преди и след max pooling.

In [ ]:
top_edges = np.maximum(convolve(digit_image, horizontal_edges), 0)
pooled = top_edges.reshape(7, 4, 7, 4).max(axis=(1, 3))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), layout="constrained")
axes[0].imshow(digit_image, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
axes[0].set_title(f"Изображение, етикет {y_test[0]}")
axes[1].imshow(top_edges, cmap="Reds", vmin=0, vmax=3, interpolation="nearest")
axes[1].set_title("Горни ръбове, 28x28")
axes[2].imshow(pooled, cmap="Reds", vmin=0, vmax=3, interpolation="nearest")
axes[2].set_title("След max pooling, 7x7")
for ax in axes:
    ax.set(xticks=[], yticks=[])
plt.show()

8 карти по 49 числа дават 392 признака вместо 784 пиксела. Обучението и
проверката са като в Част 4: същото разделяне и `Perceptron(random_state=42)`.
`correlate2d` от библиотеката SciPy прави същото като функцията `convolve`,
но е много по-бърза. Първият ред с `print` проверява това.

In [ ]:
from scipy.signal import correlate2d

diagonal_1 = np.array([[ 0,  1,  1],
                       [-1,  0,  1],
                       [-1, -1,  0]])
diagonal_2 = np.array([[ 1,  1,  0],
                       [ 1,  0, -1],
                       [ 0, -1, -1]])
edge_filters = [horizontal_edges, vertical_edges, diagonal_1, diagonal_2]
print("correlate2d дава същата карта като convolve:",
      np.allclose(correlate2d(digit_image, horizontal_edges, mode="same"),
                  convolve(digit_image, horizontal_edges)))


def conv_features(imgs):
    """8 карти на ръбовете за всяко изображение, всяка намалена до 7x7."""
    rows = []
    for img in imgs:
        parts = []
        for kernel in edge_filters:
            feature_map = correlate2d(img, kernel, mode="same")
            for half in (np.maximum(feature_map, 0), np.maximum(-feature_map, 0)):
                parts.append(half.reshape(7, 4, 7, 4).max(axis=(1, 3)).reshape(-1))
        rows.append(np.concatenate(parts))
    return np.array(rows)


conv_train = conv_features(train_images)
conv_test = conv_features(test_images)
print("Признаци:", conv_train.shape[1], "вместо", X_train.shape[1])
conv_model = Perceptron(random_state=42)
conv_model.fit(conv_train, y_train)
conv_predictions = conv_model.predict(conv_test)
print("Пиксели:", int((predictions == y_test).sum()), "от", len(y_test))
print("Конволюционни признаци:", int((conv_predictions == y_test).sum()), "от", len(y_test))

Как се справят двата модела с изместени цифри? Всички тестови изображения
се изместват с три колони надясно, както в т. 10, и се подават на двата
модела без ново обучение.

In [ ]:
shifted_test_images = np.zeros_like(test_images)
shifted_test_images[:, :, 3:] = test_images[:, :, :-3]
pixel_shift_predictions = model.predict(shifted_test_images.reshape(len(shifted_test_images), -1))
conv_shift_predictions = conv_model.predict(conv_features(shifted_test_images))
print("Изместени, пиксели:", int((pixel_shift_predictions == y_test).sum()), "от", len(y_test))
print("Изместени, конволюционни признаци:", int((conv_shift_predictions == y_test).sum()),
      "от", len(y_test))

Класификаторът е същият Perceptron, промениха се само признаците. Вместо
отделни пиксели моделът получава информация за ръбове в малки области, а
max pooling прави признаците по-малко чувствителни към изместване. Затова
моделът разпознава повече тестови цифри и много повече изместени. Това е
пример за конструиране на признаци: по-добрите признаци помагат, без да се
сменя моделът.

В истинската CNN филтрите не се задават ръчно, а се научават заедно с
класификатора, и слоевете са няколко. Върху пълния MNIST (60 000
изображения за обучение) такива мрежи грешат в по-малко от 1% от случаите.

<a id="obobshtenie"></a>
## Обобщение

**От картинка до предсказание:**

1. **Зареждане:** PNG файловете се откриват от папки с `rglob` и се зареждат
   с `imread`. Етикетът се извлича от името на папката.
2. **Признаци:** `reshape` подрежда 28x28 пиксела в ред от 784 числа.
   Всеки пиксел е един признак. Операцията е обратима. Преди това
   изображенията може да се подготвят: нормализация, изрязване, центриране.
3. **Разделяне:** `train_test_split` отделя примери за обучение и за проверка.
   Моделът учи от едните и се проверява върху другите.
4. **Обучение:** `fit` научава коефициенти, които свързват стойностите на
   пикселите с класа. `predict` използва тези коефициенти за нови картинки.
5. **Оценяване:** Точността казва колко често моделът е прав. Сравнението
   с базов модел показва дали резултатът е по-добър от случайното познаване.
   Матрицата на объркванията показва кои класове се бъркат помежду си.

**Какво може и какво не може моделът:**

- Моделът работи с числа, не с визуални форми. Не знае, че пикселите
  образуват картинка. Промяна на позицията може да промени предсказанието.
- Моделът не използва съседството на пикселите. Ако всички изображения се
  разбъркат по един и същ начин при обучението и при проверката, резултатът
  е същият. Ако се разбъркат само при проверката, моделът греши почти навсякъде.
- Конволюцията работи със съседни пиксели и затова зависи от подредбата им.
  Така CNN могат да откриват ръбове и форми, а не само отделни тъмни пиксели.
  Дори ръчни филтри с max pooling подобряват същия Perceptron и го правят
  по-устойчив на изместване.
- Класификаторът винаги избира измежду познатите класове, дори ако входът
  е празно изображение или нещо, което не е цифра.

<a id="zadacha"></a>
## Задача

В папката `data/shapes_png/` има PNG изображения на четири геометрични
фигури: кръг, квадрат, звезда и триъгълник. Файловете са подредени в папки
по клас, точно както цифрите. Всяко изображение е 28x28 пиксела, сиво.

Задачата е да се приложи целият *pipeline* от упражнението: зареждане от
файлове, подготовка на признаци, обучение и оценяване.

### 1. Зареждане на данните

Заредете файловете от `data/shapes_png/` по същия начин, по който заредихме
цифрите. Създайте `DataFrame` с колони `filepath` и `label`. Заредете
изображенията в масив. Етикетът е името на папката (текст, не число).
Покажете по два примера за всеки клас.

In [ ]:
# SHAPES_DIR = Path("data/shapes_png")
# shape_paths = sorted(SHAPES_DIR.rglob("*.png"))
# print(f"Намерени файлове: {len(shape_paths)}")
# shape_df = pd.DataFrame({...})
# shape_images = np.array([plt.imread(str(p)) for p in shape_paths])
# shape_labels = ...
# print(shape_df["label"].value_counts().sort_index())
# показване на по два примера за всеки клас

### 2. Обучение и оценяване

Подредете пикселите в ред от 784 признака. Разделете данните с
`train_test_split` (80/20, `random_state=42`, `stratify`).
Обучете `Perceptron(random_state=42)` и направете предсказания
върху тестовата част. Отпечатайте броя правилно разпознати фигури.

In [ ]:
# shape_X = shape_images.reshape(len(shape_images), -1)
# ... train_test_split ...
# shape_model = Perceptron(random_state=42)
# shape_model.fit(...)
# shape_preds = shape_model.predict(...)
# print("Правилно разпознати:", ...)

### 3. Матрица на объркванията

Постройте матрица на объркванията за фигурите. Кои два класа моделът
бърка най-често? Защо? Покажете два грешно разпознати примера.

In [ ]:
# shape_cm = confusion_matrix(...)
# показване на матрицата
# грешно разпознати примери